### گام اول : پیش پردازش و پرهیز از نشت داده

In [1]:
import pandas as pd

Full_data = pd.read_csv("austin_commute_raw.csv")

### 1.

In [2]:
Full_data["dispatched_on"] = pd.to_datetime(
    Full_data["dispatched_on"], utc=True, errors="coerce"
)

Full_data["driver_reached_on"] = pd.to_datetime(
    Full_data["driver_reached_on"], utc=True, errors="coerce"
)

Full_data["time_diff"] = (
    Full_data["driver_reached_on"] - Full_data["dispatched_on"]
).dt.total_seconds() / 60

Full_data["on_time"] = pd.NA

valid = Full_data["time_diff"].notna()

Full_data.loc[valid, "on_time"] = (
    Full_data.loc[valid, "time_diff"] <= 5
).astype(int)

class_counts = Full_data.loc[valid, "on_time"].value_counts().sort_index()
class_percentages = (
    Full_data.loc[valid, "on_time"].value_counts(normalize=True).sort_index() * 100
)

print("Class counts:")
print(class_counts)

print("\nClass percentages:")
print(class_percentages.round(2))

print("\nMissing target values:", Full_data["on_time"].isna().sum())


Class counts:
on_time
0    62849
1    87013
Name: count, dtype: int64

Class percentages:
on_time
0    41.94
1    58.06
Name: proportion, dtype: float64

Missing target values: 138


### 2.

In [3]:

import pandas as pd
import ast
import re

columns = [
    "year",
    "make",
    "model",
    "color",
    "requested_car_category",
    "status"
]

unique_before = Full_data[columns].nunique(dropna=True)

def decode_bytes(value):
    if pd.isna(value):
        return value

    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")

    if isinstance(value, str):
        value = value.strip()

        if value.startswith("b'") or value.startswith('b"'):
            try:
                decoded = ast.literal_eval(value)
                if isinstance(decoded, bytes):
                    return decoded.decode("utf-8", errors="replace")
            except (ValueError, SyntaxError):
                pass

        return value

    return value



for col in columns:
    Full_data[col] = Full_data[col].apply(decode_bytes)


Full_data["year"] = pd.to_numeric(
    Full_data["year"], errors="coerce"
).astype("Int64")


text_columns = [
    "make",
    "model",
    "color",
    "requested_car_category",
    "status"
]

for col in text_columns:
    Full_data[col] = Full_data[col].apply(
        lambda x: re.sub(r"\s+", " ", x).strip().lower()
        if isinstance(x, str) else x
    )


make_mapping = {
    "chevy": "chevrolet",
    "mercedes benz": "mercedes-benz",
    "merceded benz": "mercedes-benz",
    "mercedes": "mercedes-benz",
    "infinity": "infiniti",
    "toytoa": "toyota",
    "volkwagen": "volkswagen",
    "licoln": "lincoln",
    "tessla": "tesla",
    "mini cooper": "mini",
    "sienna": "toyota"
}

Full_data["make"] = Full_data["make"].replace(make_mapping)


unique_after = Full_data[columns].nunique(dropna=True)

comparison = pd.DataFrame({
    "Unique Before": unique_before,
    "Unique After": unique_after,
    "Difference": unique_before - unique_after
})

print(comparison)
print("on_time" in Full_data.columns)
print(Full_data["on_time"].value_counts(dropna=False))

                        Unique Before  Unique After  Difference
year                               18            18           0
make                               62            40          22
model                             424           410          14
color                              32            24           8
requested_car_category              5             5           0
status                              2             2           0
True
on_time
1       87013
0       62849
<NA>      138
Name: count, dtype: int64


### 3.

In [4]:
missing_count = Full_data.isna().sum()
missing_percent = Full_data.isna().mean() * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
})

missing_summary = missing_summary.sort_values(
    by="Missing Percent", ascending=False
)

print(missing_summary.round(2))

# Compare missing values in latitude and longitude
print("\nDispatch location comparison:")

print(
    missing_summary.loc[
        ["dispatch_location_lat", "dispatch_location_long"]
    ]
)

                           Missing Count  Missing Percent
driving_distance_to_rider          30752            20.50
driver_rating                      13896             9.26
esimtated_time_arrive               7869             5.25
rider_rating                        2629             1.75
driving_time_to_rider               2198             1.47
time_diff                            138             0.09
driver_reached_on                    138             0.09
on_time                              138             0.09
driver_accepted_on                    10             0.01
RIDE_ID                                0             0.00
created_date                           0             0.00
requested_car_category                 0             0.00
rider_id                               0             0.00
surge_factor                           0             0.00
start_location_lat                     0             0.00
completed_on                           0             0.00
start_location

### 4.

In [ ]:
drop_columns = [
    "driver_accepted_on",
    "driver_reached_on",
    "started_on",
    "completed_on",
    "driving_time_to_rider",
    "esimtated_time_arrive",
    "status"
]

Final_data = Full_data.drop(
    columns=drop_columns,
    errors="ignore"
)

Final_data = Final_data.dropna(subset=["on_time"])

X = Final_data.drop(columns=["on_time"])
y = Final_data["on_time"].astype(int)

print("Final attributes:")
print(X.columns.tolist())
print("Number of attributes:", X.shape[1])

Final attributes:
['RIDE_ID', 'created_date', 'start_location_lat', 'start_location_long', 'driver_rating', 'rider_rating', 'requested_car_category', 'surge_factor', 'rider_id', 'dispatched_on', 'dispatch_location_lat', 'dispatch_location_long', 'base_fare', 'rate_per_mile', 'rate_per_minute', 'driving_distance_to_rider', 'driver_id', 'car_id', 'make', 'model', 'year', 'color', 'car_categories_bitmask', 'pickup_zone', 'downtown_km', 'time_diff']
Number of attributes: 26
